<a href="https://colab.research.google.com/github/fidlarsyn/Scikit-learn-Cookbook--O-Reilly-/blob/main/13_Deploying_scikit_learn_Models_in_Production.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Overview of model deployment**
Penyebaran (deployment) adalah proses memindahkan model dari lingkungan pengembangan ke lingkungan produksi agar sistem atau pengguna akhir dapat mengakses prediksinya. Proses ini melibatkan pengemasan model menjadi artefak yang andal, memastikan kecepatan respons (latency) yang rendah, dan throughput yang tinggi.
Langkah Umum:
1. Melatih model di lingkungan pengembangan.
2. Menyimpan model ke dalam format file (artefak).
3. Memuat artefak tersebut di server produksi untuk melayani permintaan prediksi.


# **Serialization and persistence techniques**
Agar model bisa disimpan dan digunakan kembali tanpa melatih ulang, kita menggunakan teknik serialisasi yang mengubah objek model menjadi aliran biner (byte stream).
* **joblib**: Sangat direkomendasikan untuk scikit-learn karena lebih efisien dalam menangani array NumPy yang besar dibandingkan pickle standar.
* **pickle**: Modul bawaan Python untuk serialisasi objek, namun bisa bermasalah dengan data numerik yang sangat besar.

**Contoh Kode Serialisasi dengan joblib:**

In [ ]:
from joblib import dump, load
import numpy as np

# 1. Menyimpan model ke file
dump(clf, "model.joblib")

# 2. Memuat model kembali di lingkungan baru
model_siap_pakai = load("model.joblib")

# 3. Melakukan prediksi
data_baru = np.random.rand(1, 20)
hasil = model_siap_pakai.predict(data_baru)

# **Scaling models for production**
Di lingkungan produksi, model harus mampu menangani dataset besar atau permintaan prediksi yang tinggi secara bersamaan.
* **Paralelisme (n_jobs)**: Banyak estimator scikit-learn mendukung parameter n_jobs=-1 untuk menggunakan semua inti CPU yang tersedia saat pelatihan atau prediksi.
* **Batch Serving**: Melakukan prediksi pada sekumpulan data sekaligus (batch) jauh lebih efisien daripada memanggil prediksi satu per satu karena optimasi aljabar linear pada perangkat keras.
* **Dask**: Dapat digunakan sebagai backend untuk mendistribusikan beban kerja ke klaster komputer yang lebih besar.


# **Monitoring and updating deployed models**
Kinerja model pasti akan menurun seiring waktu karena perubahan distribusi data input (disebut sebagai drift). Strategi Continuous Training (CT) diperlukan untuk memantau penurunan ini dan melatih ulang model secara otomatis.

**Contoh Kode Pemantauan Akurasi Batch:**

In [ ]:
from sklearn.linear_model import SGDClassifier

# Menggunakan partial_fit untuk pembelajaran bertahap (incremental)
clf = SGDClassifier(loss="log_loss", warm_start=True)
clf.partial_fit(X_train, y_train, classes=np.unique(y_train))

# Pantau akurasi pada batch data yang baru masuk
for xb, yb in zip(stream_batches, stream_labels):
    skor = clf.score(xb, yb)
    if skor < 0.8: # Threshold akurasi
        print("Kinerja menurun, memicu pelatihan ulang!")
        clf.partial_fit(xb, yb)

# **Managing the model life cycle**
Pengelolaan siklus hidup model memastikan model tetap andal, dapat direproduksi, dan mudah dipelihara. Ini adalah bagian dari praktik MLOps.
* **Versioning**: Memberikan versi pada setiap file model (misal: v1.0, v1.1) agar bisa dilakukan rollback jika versi terbaru bermasalah.
* **Metadata**: Menyimpan informasi lingkungan seperti versi pustaka scikit-learn dan NumPy saat model dilatih.
* **Snapshot Validasi**: Menyimpan sebagian kecil data uji beserta hasil prediksinya untuk memverifikasi konsistensi model setelah pembaruan sistem.

# **Setting up deployment pipelines**
Menggabungkan langkah prapemrosesan dan model ke dalam satu objek Pipeline adalah praktik terbaik untuk memastikan konsistensi antara tahap pelatihan dan produksi.

**Contoh Kode Pipeline dengan Cek Validasi Otomatis:**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Membundel transformasi dan model
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression())
])
pipe.fit(X_train, y_train)

# Serialisasi seluruh alur kerja
dump(pipe, "pipeline_produksi.joblib")

# Simulasi CI/CD: Hanya deploy jika akurasi data uji > 80%
model_prod = load("pipeline_produksi.joblib")
acc = model_prod.score(X_test, y_test)

if acc > 0.8:
    print("Otomatisasi penyebaran diizinkan.")
else:
    print("Hentikan penyebaran dan tinjau kembali!")